# Proxy5 — Deep Learning Model Comparison for Oil Recovery Prediction

**Objective:** Compare four deep learning architectures for predicting oil recovery factor from the Proxy5 dataset.

| # | Model | Key Strength |
|---|-------|--------------|
| 1 | **MLP** — Multi-Layer Perceptron | Universal function approximator; fast baseline |
| 2 | **LSTM** — Long Short-Term Memory | Captures sequential/temporal dependencies |
| 3 | **CNN-LSTM** — 1-D Conv + LSTM | Local feature extraction + temporal memory |
| 4 | **PINN** — Physics-Informed Neural Network | Embeds Buckley–Leverett physics (monotonicity) as a loss term |

**Data file:** `Proxy5.csv` (3 306 rows × 15 columns)  
**Target:** `Oil_recovery_factor (%)`  
**Physics reference:** Liu et al. *Physics of Fluids* 37 036622 (2025)

## 1. Imports & Reproducibility

In [ ]:
import warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})
sns.set_palette('tab10')

COLORS = {'MLP': '#1f77b4', 'LSTM': '#ff7f0e', 'CNN-LSTM': '#2ca02c', 'PINN': '#d62728'}

## 2. Load Data

In [ ]:
df = pd.read_csv('Proxy5.csv', encoding='latin1')
print(f'Shape: {df.shape}')
print(df.dtypes)
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# ----- 3.1  Summary statistics -----
print(df.describe().round(4).to_string())
print('\nMissing values:')
print(df.isnull().sum())

In [ ]:
# ----- 3.2  Target distribution -----
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df['Oil_recovery_factor (%)'].dropna(), bins=60,
             color='steelblue', edgecolor='white', linewidth=0.4)
axes[0].set_xlabel('Oil Recovery Factor (%)')
axes[0].set_ylabel('Count')
axes[0].set_title('Target Distribution')

axes[1].boxplot(df['Oil_recovery_factor (%)'].dropna(), vert=True,
                patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].set_ylabel('Oil Recovery Factor (%)')
axes[1].set_title('Target Box Plot')

plt.suptitle('Target Variable: Oil Recovery Factor', fontsize=13)
plt.tight_layout()
plt.savefig('plot_01_target_dist.png', bbox_inches='tight')
plt.show()

In [ ]:
# ----- 3.3  Feature distributions -----
feature_cols = [c for c in df.columns if c != 'Oil_recovery_factor (%)']
n_feat = len(feature_cols)
ncols = 4
nrows = (n_feat + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3))
for ax, col in zip(axes.flat, feature_cols):
    ax.hist(df[col].dropna(), bins=40, color='teal',
            edgecolor='white', linewidth=0.3, alpha=0.8)
    ax.set_xlabel(col, fontsize=8)
    ax.set_ylabel('Count', fontsize=8)
    ax.tick_params(labelsize=7)
# hide unused axes
for ax in axes.flat[n_feat:]:
    ax.set_visible(False)

plt.suptitle('Input Feature Distributions', fontsize=13)
plt.tight_layout()
plt.savefig('plot_02_feature_dist.png', bbox_inches='tight')
plt.show()

In [ ]:
# ----- 3.4  Correlation heatmap -----
corr = df.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax,
            annot_kws={'size': 7}, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.savefig('plot_03_correlation.png', bbox_inches='tight')
plt.show()

In [ ]:
# ----- 3.5  Feature vs target scatter (top 6 by absolute correlation) -----
target = 'Oil_recovery_factor (%)'
top6 = corr[target].drop(target).abs().nlargest(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, top6):
    sub = df[[col, target]].dropna()
    ax.scatter(sub[col], sub[target], alpha=0.3, s=6, color='steelblue')
    ax.set_xlabel(col, fontsize=8)
    ax.set_ylabel(target, fontsize=8)
    r = sub.corr().iloc[0, 1]
    ax.set_title(f'r = {r:.3f}', fontsize=9)

plt.suptitle('Top-6 Features vs Oil Recovery Factor', fontsize=13)
plt.tight_layout()
plt.savefig('plot_04_scatter_top6.png', bbox_inches='tight')
plt.show()

## 4. Data Preprocessing

In [ ]:
TARGET = 'Oil_recovery_factor (%)'
FEATURE_COLS = [c for c in df.columns if c != TARGET]

# Drop rows where target is missing; impute single missing feature rows with median
df_clean = df.dropna(subset=[TARGET]).copy()
df_clean[FEATURE_COLS] = df_clean[FEATURE_COLS].fillna(df_clean[FEATURE_COLS].median())

print(f'Rows after cleaning: {len(df_clean)}  (dropped {len(df) - len(df_clean)} with missing target)')

X_all = df_clean[FEATURE_COLS].values.astype(np.float32)
y_all = df_clean[[TARGET]].values.astype(np.float32)

X_train, X_temp, y_train, y_temp = train_test_split(X_all, y_all,
                                                      test_size=0.30, random_state=SEED)
X_val,   X_test, y_val,   y_test  = train_test_split(X_temp, y_temp,
                                                      test_size=0.50, random_state=SEED)

print(f'Train: {X_train.shape[0]}  Val: {X_val.shape[0]}  Test: {X_test.shape[0]}')

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train_s = scaler_X.fit_transform(X_train)
y_train_s = scaler_y.fit_transform(y_train)
X_val_s   = scaler_X.transform(X_val)
y_val_s   = scaler_y.transform(y_val)
X_test_s  = scaler_X.transform(X_test)
y_test_s  = scaler_y.transform(y_test)

def to_t(*arrs):
    return [torch.tensor(a, dtype=torch.float32).to(DEVICE) for a in arrs]

Xt_tr, yt_tr = to_t(X_train_s, y_train_s)
Xt_va, yt_va = to_t(X_val_s,   y_val_s)
Xt_te, yt_te = to_t(X_test_s,  y_test_s)

BATCH = 256
train_loader = DataLoader(TensorDataset(Xt_tr, yt_tr), batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(TensorDataset(Xt_va, yt_va), batch_size=BATCH)

N_FEAT = X_train_s.shape[1]
print(f'n_features = {N_FEAT}')

## 5. Model Architectures

In [ ]:
# =========================================================================
# MODEL 1: MLP
# =========================================================================
class MLP(nn.Module):
    def __init__(self, in_features, hidden=(128, 256, 256, 128), dropout=0.15):
        super().__init__()
        layers = []
        prev = in_features
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers += [nn.Linear(prev, 1), nn.Sigmoid()]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


# =========================================================================
# MODEL 2: LSTM
# =========================================================================
class LSTMModel(nn.Module):
    """Each feature treated as one time-step (sequence length = n_features)."""
    def __init__(self, n_features, hidden=128, n_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden,
                            num_layers=n_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Sequential(
            nn.Linear(hidden, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Sigmoid()
        )

    def forward(self, x):
        x = x.unsqueeze(-1)        # (B, n_features, 1)
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


# =========================================================================
# MODEL 3: CNN-LSTM
# =========================================================================
class CNNLSTMModel(nn.Module):
    def __init__(self, n_features, cnn_channels=64, kernel=3,
                 lstm_hidden=128, n_lstm_layers=2, dropout=0.2):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, cnn_channels, kernel_size=kernel, padding=kernel // 2),
            nn.ReLU(),
            nn.Conv1d(cnn_channels, cnn_channels, kernel_size=kernel, padding=kernel // 2),
            nn.ReLU(),
        )
        self.lstm = nn.LSTM(input_size=cnn_channels, hidden_size=lstm_hidden,
                            num_layers=n_lstm_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Sequential(
            nn.Linear(lstm_hidden, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Sigmoid()
        )

    def forward(self, x):
        x = x.unsqueeze(1)          # (B, 1, n_features)
        x = self.conv(x)            # (B, cnn_channels, n_features)
        x = x.permute(0, 2, 1)     # (B, n_features, cnn_channels)
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


# =========================================================================
# MODEL 4: PINN
# Physics loss: penalise negative dRF/d(APV)  (BL monotonicity)
#               penalise positive d²RF/d(APV)² (BL concavity)
# APV (Accessible Pore Volume) is the index-0 feature after scaling.
# =========================================================================
class PINNModel(nn.Module):
    def __init__(self, in_features, hidden=(128, 256, 256, 128), dropout=0.1):
        super().__init__()
        layers = []
        prev = in_features
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.Tanh()]
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers += [nn.Linear(prev, 1), nn.Sigmoid()]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

    def physics_loss(self, x_in, apv_col=0):
        """Buckley-Leverett monotonicity + concavity constraints w.r.t. APV."""
        x = x_in.clone().requires_grad_(True)
        rf = self.net(x)
        g1 = torch.autograd.grad(rf.sum(), x, create_graph=True)[0]
        drf_dapv = g1[:, apv_col:apv_col + 1]
        g2 = torch.autograd.grad(drf_dapv.sum(), x, create_graph=True)[0]
        d2rf = g2[:, apv_col:apv_col + 1]
        loss_mono    = torch.relu(-drf_dapv).pow(2).mean()
        loss_concave = torch.relu(d2rf).pow(2).mean()
        return loss_mono + loss_concave


print('All model classes defined.')

## 6. Training Utilities

In [ ]:
def train_model(model, train_loader, val_loader, epochs=300,
                lr=5e-4, weight_decay=1e-4, patience=30,
                pinn_lambda=0.0, pinn_apv_col=0):
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=10, factor=0.5, verbose=False)
    criterion = nn.MSELoss()
    history   = {'train_loss': [], 'val_loss': [], 'epoch': []}
    best_val, best_state, no_imp = np.inf, None, 0

    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train()
        tr = 0.0
        for xb, yb in train_loader:
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            if pinn_lambda > 0:
                loss = loss + pinn_lambda * model.physics_loss(xb, pinn_apv_col)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tr += loss.item() * xb.size(0)
        tr /= len(train_loader.dataset)

        model.eval()
        va = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                va += criterion(model(xb), yb).item() * xb.size(0)
        va /= len(val_loader.dataset)

        scheduler.step(va)
        history['train_loss'].append(tr)
        history['val_loss'].append(va)
        history['epoch'].append(ep)

        if va < best_val:
            best_val = va
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1
            if no_imp >= patience:
                print(f'  Early stop at epoch {ep}  (best val MSE={best_val:.6f})')
                break

    model.load_state_dict(best_state)
    print(f'  Done in {time.time()-t0:.1f}s | best val MSE={best_val:.6f}')
    return history


def evaluate(model, Xt, yt_scaled, scaler_y):
    model.eval()
    with torch.no_grad():
        yp_s = model(Xt).cpu().numpy()
    yp = scaler_y.inverse_transform(yp_s)
    yt = scaler_y.inverse_transform(yt_scaled.cpu().numpy())
    return (yp.flatten(), yt.flatten(),
            {'R2':   r2_score(yt, yp),
             'RMSE': np.sqrt(mean_squared_error(yt, yp)),
             'MAE':  mean_absolute_error(yt, yp)})


results = {}
print('Utilities ready.')

## 7. Train All Four Models

In [ ]:
# ------------------------------------------------------------------
# 7.1  MLP
# ------------------------------------------------------------------
print('=' * 55)
print('Training MLP …')
mlp = MLP(N_FEAT).to(DEVICE)
print(f'  Parameters: {sum(p.numel() for p in mlp.parameters()):,}')
hist_mlp = train_model(mlp, train_loader, val_loader)
yp_mlp, yt, m_mlp = evaluate(mlp, Xt_te, yt_te, scaler_y)
results['MLP'] = {'metrics': m_mlp, 'history': hist_mlp, 'y_pred': yp_mlp, 'y_true': yt}
print(f'  R²={m_mlp["R2"]:.4f}  RMSE={m_mlp["RMSE"]:.4f}  MAE={m_mlp["MAE"]:.4f}')

In [ ]:
# ------------------------------------------------------------------
# 7.2  LSTM
# ------------------------------------------------------------------
print('=' * 55)
print('Training LSTM …')
lstm = LSTMModel(N_FEAT).to(DEVICE)
print(f'  Parameters: {sum(p.numel() for p in lstm.parameters()):,}')
hist_lstm = train_model(lstm, train_loader, val_loader)
yp_lstm, yt, m_lstm = evaluate(lstm, Xt_te, yt_te, scaler_y)
results['LSTM'] = {'metrics': m_lstm, 'history': hist_lstm, 'y_pred': yp_lstm, 'y_true': yt}
print(f'  R²={m_lstm["R2"]:.4f}  RMSE={m_lstm["RMSE"]:.4f}  MAE={m_lstm["MAE"]:.4f}')

In [ ]:
# ------------------------------------------------------------------
# 7.3  CNN-LSTM
# ------------------------------------------------------------------
print('=' * 55)
print('Training CNN-LSTM …')
cnn_lstm = CNNLSTMModel(N_FEAT).to(DEVICE)
print(f'  Parameters: {sum(p.numel() for p in cnn_lstm.parameters()):,}')
hist_cnn = train_model(cnn_lstm, train_loader, val_loader)
yp_cnn, yt, m_cnn = evaluate(cnn_lstm, Xt_te, yt_te, scaler_y)
results['CNN-LSTM'] = {'metrics': m_cnn, 'history': hist_cnn, 'y_pred': yp_cnn, 'y_true': yt}
print(f'  R²={m_cnn["R2"]:.4f}  RMSE={m_cnn["RMSE"]:.4f}  MAE={m_cnn["MAE"]:.4f}')

In [ ]:
# ------------------------------------------------------------------
# 7.4  PINN  (λ=0.05 weights the BL physics residual)
# ------------------------------------------------------------------
print('=' * 55)
print('Training PINN …')
pinn = PINNModel(N_FEAT).to(DEVICE)
print(f'  Parameters: {sum(p.numel() for p in pinn.parameters()):,}')
# APV is column 0 in FEATURE_COLS — index preserved after MinMaxScaler
hist_pinn = train_model(pinn, train_loader, val_loader,
                         pinn_lambda=0.05, pinn_apv_col=0)
yp_pinn, yt, m_pinn = evaluate(pinn, Xt_te, yt_te, scaler_y)
results['PINN'] = {'metrics': m_pinn, 'history': hist_pinn, 'y_pred': yp_pinn, 'y_true': yt}
print(f'  R²={m_pinn["R2"]:.4f}  RMSE={m_pinn["RMSE"]:.4f}  MAE={m_pinn["MAE"]:.4f}')

## 8. Results & Plots

In [ ]:
# ----- 8.1  Metrics table -----
model_names = list(results.keys())
model_objs  = {'MLP': mlp, 'LSTM': lstm, 'CNN-LSTM': cnn_lstm, 'PINN': pinn}
colors_list = [COLORS[n] for n in model_names]

rows = []
for name in model_names:
    m = results[name]['metrics']
    rows.append({'Model': name,
                 'R²':    round(m['R2'],   4),
                 'RMSE':  round(m['RMSE'], 4),
                 'MAE':   round(m['MAE'],  4),
                 'Params': sum(p.numel() for p in model_objs[name].parameters())})

df_metrics = (pd.DataFrame(rows)
               .sort_values('R²', ascending=False)
               .reset_index(drop=True))
df_metrics.index += 1
print('\n===  Test-Set Performance  ===')
print(df_metrics.to_string())

In [ ]:
# ----- 8.2  Training & validation loss curves -----
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, name in zip(axes, model_names):
    h = results[name]['history']
    ax.semilogy(h['epoch'], h['train_loss'], color=COLORS[name], label='Train')
    ax.semilogy(h['epoch'], h['val_loss'],   color=COLORS[name], linestyle='--', label='Val')
    ax.set_title(name); ax.set_xlabel('Epoch'); ax.set_ylabel('MSE (log)')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.suptitle('Training & Validation Loss Curves', fontsize=13)
plt.tight_layout()
plt.savefig('plot_05_loss_curves.png', bbox_inches='tight')
plt.show()

In [ ]:
# ----- 8.3  Parity plots (predicted vs actual) -----
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, name in zip(axes, model_names):
    yt = results[name]['y_true']
    yp = results[name]['y_pred']
    m  = results[name]['metrics']
    lim = [min(yt.min(), yp.min()) - 0.05,
           max(yt.max(), yp.max()) + 0.05]
    ax.scatter(yt, yp, alpha=0.25, s=5, color=COLORS[name])
    ax.plot(lim, lim, 'k--', linewidth=0.8)
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel('Actual RF (%)')
    ax.set_ylabel('Predicted RF (%)')
    ax.set_title(f'{name}\nR²={m["R2"]:.4f}  RMSE={m["RMSE"]:.4f}')
plt.suptitle('Parity Plots — Predicted vs Actual Oil Recovery Factor', fontsize=13)
plt.tight_layout()
plt.savefig('plot_06_parity.png', bbox_inches='tight')
plt.show()

In [ ]:
# ----- 8.4  Residual distributions -----
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, name in zip(axes, model_names):
    res = results[name]['y_pred'] - results[name]['y_true']
    ax.hist(res, bins=50, color=COLORS[name], edgecolor='white', linewidth=0.3, alpha=0.85)
    ax.axvline(0, color='black', linestyle='--', linewidth=1)
    ax.axvline(res.mean(), color='red', linewidth=1,
               label=f'Mean={res.mean():.4f}')
    ax.set_xlabel('Residual (Pred − True)')
    ax.set_ylabel('Count')
    ax.set_title(f'{name} Residuals')
    ax.legend(fontsize=8)
plt.suptitle('Residual Distributions', fontsize=13)
plt.tight_layout()
plt.savefig('plot_07_residuals.png', bbox_inches='tight')
plt.show()

In [ ]:
# ----- 8.5  Residuals vs actual (heteroscedasticity check) -----
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, name in zip(axes, model_names):
    res = results[name]['y_pred'] - results[name]['y_true']
    ax.scatter(results[name]['y_true'], res, alpha=0.2, s=5, color=COLORS[name])
    ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
    ax.set_xlabel('Actual RF (%)')
    ax.set_ylabel('Residual')
    ax.set_title(name)
plt.suptitle('Residuals vs Actual — Heteroscedasticity Check', fontsize=13)
plt.tight_layout()
plt.savefig('plot_08_residuals_vs_actual.png', bbox_inches='tight')
plt.show()

In [ ]:
# ----- 8.6  CDF of absolute errors -----
fig, ax = plt.subplots(figsize=(8, 5))
for name in model_names:
    ae = np.sort(np.abs(results[name]['y_pred'] - results[name]['y_true']))
    cdf = np.arange(1, len(ae) + 1) / len(ae)
    ax.plot(ae, cdf, linewidth=2, label=name, color=COLORS[name])
ax.axvline(0.05, color='gray', linestyle='--', linewidth=0.8, label='5% error')
ax.set_xlabel('Absolute Error (RF %)')
ax.set_ylabel('Cumulative Fraction')
ax.set_title('CDF of Absolute Prediction Errors')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('plot_09_cdf_errors.png', bbox_inches='tight')
plt.show()

In [ ]:
# ----- 8.7  Metric bar charts -----
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, metric in zip(axes, ['R2', 'RMSE', 'MAE']):
    vals = [results[n]['metrics'][metric] for n in model_names]
    bars = ax.bar(model_names, vals, color=colors_list, edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(vals) * 0.01,
                f'{v:.4f}', ha='center', va='bottom', fontsize=9)
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} — Test Set')
    if metric == 'R2':
        ax.set_ylim(0, 1.1)
plt.suptitle('Model Performance Comparison', fontsize=13)
plt.tight_layout()
plt.savefig('plot_10_bar_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
# ----- 8.8  Radar chart -----
max_rmse = max(results[n]['metrics']['RMSE'] for n in model_names)
max_mae  = max(results[n]['metrics']['MAE']  for n in model_names)
max_par  = max(sum(p.numel() for p in model_objs[n].parameters()) for n in model_names)

def radar_scores(name):
    m = results[name]['metrics']
    np_ = sum(p.numel() for p in model_objs[name].parameters())
    return [
        m['R2'],
        1 - m['RMSE'] / max_rmse,
        1 - m['MAE']  / max_mae,
        1 - np_ / max_par,
    ]

labels = ['R²', '1-RMSE\n(norm)', '1-MAE\n(norm)', 'Efficiency\n(fewer params)']
angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
for name in model_names:
    v = radar_scores(name) + radar_scores(name)[:1]
    ax.plot(angles, v, '-o', linewidth=2, color=COLORS[name], label=name)
    ax.fill(angles, v, alpha=0.10, color=COLORS[name])
ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylim(0, 1)
ax.set_title('Multi-Metric Radar Chart', fontsize=13, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=10)
plt.tight_layout()
plt.savefig('plot_11_radar.png', bbox_inches='tight')
plt.show()

In [ ]:
# ----- 8.9  Sensitivity: RF vs Oil Viscosity sweep -----
median_raw = df_clean[FEATURE_COLS].median().values.astype(np.float32)
median_s   = scaler_X.transform([median_raw])[0]

visc_col = FEATURE_COLS.index('Oil_viscosity (cp)')
visc_range = np.linspace(df_clean['Oil_viscosity (cp)'].min(),
                          df_clean['Oil_viscosity (cp)'].max(), 200)

X_sw = np.tile(median_s, (200, 1)).astype(np.float32)
v_min = df_clean['Oil_viscosity (cp)'].min()
v_max = df_clean['Oil_viscosity (cp)'].max()
X_sw[:, visc_col] = ((visc_range - v_min) / (v_max - v_min)).astype(np.float32)
Xs = torch.tensor(X_sw, dtype=torch.float32).to(DEVICE)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, mdl in model_objs.items():
    mdl.eval()
    with torch.no_grad():
        preds = scaler_y.inverse_transform(mdl(Xs).cpu().numpy())
    axes[0].plot(visc_range, preds, label=name, color=COLORS[name], linewidth=2)
axes[0].set_xlabel('Oil Viscosity (cp)')
axes[0].set_ylabel('Predicted RF (%)')
axes[0].set_title('Sensitivity: RF vs Oil Viscosity')
axes[0].set_xscale('log')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# Sensitivity: RF vs Permeability
perm_col = FEATURE_COLS.index('Permeability (md)')
perm_range = np.linspace(df_clean['Permeability (md)'].min(),
                          df_clean['Permeability (md)'].max(), 200)
X_sw2 = np.tile(median_s, (200, 1)).astype(np.float32)
p_min, p_max = df_clean['Permeability (md)'].min(), df_clean['Permeability (md)'].max()
X_sw2[:, perm_col] = ((perm_range - p_min) / (p_max - p_min)).astype(np.float32)
Xs2 = torch.tensor(X_sw2, dtype=torch.float32).to(DEVICE)

for name, mdl in model_objs.items():
    mdl.eval()
    with torch.no_grad():
        preds2 = scaler_y.inverse_transform(mdl(Xs2).cpu().numpy())
    axes[1].plot(perm_range, preds2, label=name, color=COLORS[name], linewidth=2)
axes[1].set_xlabel('Permeability (md)')
axes[1].set_ylabel('Predicted RF (%)')
axes[1].set_title('Sensitivity: RF vs Permeability')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.suptitle('Sensitivity Analysis (all other features at median)', fontsize=13)
plt.tight_layout()
plt.savefig('plot_12_sensitivity.png', bbox_inches='tight')
plt.show()

In [ ]:
# ----- 8.10  PINN physics compliance: dRF/d(APV) positivity -----
apv_col = FEATURE_COLS.index('APV')
apv_range = np.linspace(0, 1, 150).astype(np.float32)   # APV is already [0,1] after scaling
X_ph = np.tile(median_s, (150, 1)).astype(np.float32)
X_ph[:, apv_col] = apv_range
Xt_ph = torch.tensor(X_ph, dtype=torch.float32, requires_grad=True).to(DEVICE)

pinn.eval()
rf_ph = pinn(Xt_ph)
grad_ph = torch.autograd.grad(rf_ph.sum(), Xt_ph)[0]
drf_dapv = grad_ph[:, apv_col].detach().cpu().numpy()
rf_ph_np = scaler_y.inverse_transform(rf_ph.detach().cpu().numpy())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(apv_range, rf_ph_np, color=COLORS['PINN'], linewidth=2)
axes[0].set_xlabel('APV (scaled)')
axes[0].set_ylabel('Predicted RF (%)')
axes[0].set_title('PINN — RF vs APV (median reservoir)')
axes[0].grid(alpha=0.3)

axes[1].plot(apv_range, drf_dapv, color=COLORS['PINN'], linewidth=2)
axes[1].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[1].fill_between(apv_range, 0, drf_dapv,
                      where=drf_dapv >= 0, alpha=0.3, color='green', label='Physical (≥0)')
axes[1].fill_between(apv_range, 0, drf_dapv,
                      where=drf_dapv < 0,  alpha=0.3, color='red',   label='Unphysical (<0)')
axes[1].set_xlabel('APV (scaled)')
axes[1].set_ylabel('dRF/d(APV)')
axes[1].set_title('PINN — Monotonicity Check (BL Physics)')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

pct = np.mean(drf_dapv >= 0) * 100
print(f'PINN monotonicity compliance: {pct:.1f}% of points satisfy dRF/d(APV) ≥ 0')
plt.suptitle('PINN Physics Compliance — Buckley–Leverett Monotonicity', fontsize=13)
plt.tight_layout()
plt.savefig('plot_13_pinn_physics.png', bbox_inches='tight')
plt.show()

In [ ]:
# ----- 8.11  Convergence overlay (all models on one axis) -----
fig, ax = plt.subplots(figsize=(9, 5))
for name in model_names:
    h = results[name]['history']
    ax.semilogy(h['epoch'], h['val_loss'], linewidth=1.8,
                label=name, color=COLORS[name])
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation MSE Loss')
ax.set_title('Validation Loss Convergence — All Models')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('plot_14_convergence_overlay.png', bbox_inches='tight')
plt.show()

In [ ]:
# ----- 8.12  Summary dashboard -----
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.38, wspace=0.32)

# R²
ax1 = fig.add_subplot(gs[0, 0])
r2v = [results[n]['metrics']['R2'] for n in model_names]
bars = ax1.barh(model_names, r2v, color=colors_list, edgecolor='white')
for bar, v in zip(bars, r2v):
    ax1.text(v - 0.005, bar.get_y() + bar.get_height() / 2,
             f'{v:.4f}', ha='right', va='center', fontsize=9,
             color='white', fontweight='bold')
ax1.set_xlim(0, 1); ax1.set_xlabel('R²'); ax1.set_title('R²')

# RMSE
ax2 = fig.add_subplot(gs[0, 1])
rmv = [results[n]['metrics']['RMSE'] for n in model_names]
b2  = ax2.barh(model_names, rmv, color=colors_list, edgecolor='white')
for bar, v in zip(b2, rmv):
    ax2.text(v + max(rmv)*0.01, bar.get_y() + bar.get_height()/2,
             f'{v:.4f}', ha='left', va='center', fontsize=9)
ax2.set_xlabel('RMSE'); ax2.set_title('RMSE')

# MAE
ax3 = fig.add_subplot(gs[0, 2])
mav = [results[n]['metrics']['MAE'] for n in model_names]
b3  = ax3.barh(model_names, mav, color=colors_list, edgecolor='white')
for bar, v in zip(b3, mav):
    ax3.text(v + max(mav)*0.01, bar.get_y() + bar.get_height()/2,
             f'{v:.4f}', ha='left', va='center', fontsize=9)
ax3.set_xlabel('MAE'); ax3.set_title('MAE')

# Convergence overlay
ax4 = fig.add_subplot(gs[1, 0:2])
for name in model_names:
    h = results[name]['history']
    ax4.semilogy(h['epoch'], h['val_loss'], linewidth=1.5,
                  label=name, color=COLORS[name])
ax4.set_xlabel('Epoch'); ax4.set_ylabel('Val MSE Loss')
ax4.set_title('Validation Convergence'); ax4.legend(fontsize=9); ax4.grid(alpha=0.3)

# Metrics table
ax5 = fig.add_subplot(gs[1, 2])
ax5.axis('off')
tbl = ax5.table(
    cellText=[[n, f'{results[n]["metrics"]["R2"]:.4f}',
               f'{results[n]["metrics"]["RMSE"]:.4f}',
               f'{results[n]["metrics"]["MAE"]:.4f}']
              for n in model_names],
    colLabels=['Model', 'R²', 'RMSE', 'MAE'],
    loc='center', cellLoc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1, 2.2)
ax5.set_title('Test Set Summary', fontsize=12)

fig.suptitle('Proxy5 — DL Model Comparison for Oil Recovery Prediction\n(Proxy5.csv)',
             fontsize=14, y=1.01)
plt.savefig('plot_15_summary_dashboard.png', bbox_inches='tight')
plt.show()

## 9. Final Ranking & Conclusions

In [ ]:
print('\n' + '=' * 65)
print('   PROXY5 — FINAL MODEL RANKING (by R²)')
print('=' * 65)
print(df_metrics[['Model', 'R²', 'RMSE', 'MAE', 'Params']].to_string(index=True))
print('=' * 65)

best = df_metrics.iloc[0]
print(f'\n>> Best model : {best["Model"]}'
      f'  (R²={best["R²"]:.4f},  RMSE={best["RMSE"]:.4f},  MAE={best["MAE"]:.4f})')

print('''
NOTES
─────
• MLP      : Fastest to train; strong tabular-data baseline.
• LSTM     : Treats features as a sequence; benefits from ordering by physical meaning.
• CNN-LSTM : 1-D convolution captures local feature interactions before temporal LSTM.
• PINN     : Enforces Buckley–Leverett monotonicity (dRF/d(APV) ≥ 0) as a physics
             loss term; most robust under sparse data or out-of-distribution conditions.

Data   : Proxy5.csv  (3 306 rows × 14 features)
Physics: Liu et al. Physics of Fluids 37 036622 (2025)
''')